In [1]:
import os
import time
import joblib
import warnings
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession

In [2]:
warnings.filterwarnings('ignore', category=UserWarning)

In [ ]:
#initialize spark session (SQL Server JDBC driver)
print(f'Initializing spark session with cloud JDBC compatibility:')

spark = (SparkSession.builder
         .appName("FraudDetection")
         .config("spark.jars.packages", "com.microsoft.sqlserver:mssql-jdbc:12.8.1.jre11")
         .getOrCreate())

Initializing spark session with cloud JDBC compatibility:


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/19 15:25:00 WARN Utils: Your hostname, SivhoungdeMacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.3 instead (on interface en0)
26/06/19 15:25:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/sivhoung/.ivy2.5.2/cache
The jars for the packages stored in: /Users/sivhoung/.ivy2.5.2/jars
com.microsoft.sqlserver#mssql-jdbc added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-74463cd8-c580-423c-a4fa-6bf9c5879000;1.0
	confs: [default]
	found com.microsoft.sqlserver#mssql-jdbc;12.8.1.jre11 in central
:: resolution report :: resolve 77ms :: artifacts dl 1ms
	:: modules in use:
	com.microsoft.sqlserver#ms

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 62918)
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pyspark/accumulators.py", line 303, in handle
    poll(accum_updates)
  File "/Library/Frameworks/Pyt

In [4]:
#Extracting saved model and scalers

try:
    best_model = joblib.load('/Users/sivhoung/Fraud_detection/best_fraud_detection_model.pkl')
    scaler = joblib.load('/Users/sivhoung/Fraud_detection/preprocessing_scaler.pkl')
    print('Artifacts successfully deployed into Pyspark memory layer.')
except FileNotFoundError:
    print("Error: cannot find the file.")
    raise

Artifacts successfully deployed into Pyspark memory layer.


In [5]:
jdbc_url = "jdbc:sqlserver://paysim-db.c78a2mamcvtr.ap-southeast-2.rds.amazonaws.com:1433;databaseName=FraudDetectionDB;encrypt=true;trustServerCertificate=true;"
db_properties = {
    "user": "admin",
    "password": "GCTFFh6bOvcZMPSPmE2k",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}


In [6]:
from datetime import datetime
import traceback

target_in_table = "dbo.live_transactions"
target_out_table = "dbo.fraud_predictions"

processed_transactions = set()

print(" [INIT] PySpark AI Fraud Inference Engine is now online...")
print(" Polling Amazon RDS for new incoming financial streams...\n")

try:
    while True:
        try:
            df_live = (spark.read
                       .format("jdbc")
                       .option("url", jdbc_url)
                       .option("dbtable", target_in_table)
                       .option("user", db_properties.get("user"))
                       .option("password", db_properties.get("password"))
                       .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
                       .load()
                       .limit(500))

            pdf_live = df_live.toPandas()

            if pdf_live.empty:
                time.sleep(3)
                continue

            pdf_live['uid'] = pdf_live['nameOrig'].astype(str) + "_" + pdf_live['amount'].astype(str)
            pdf_new = pdf_live[~pdf_live['uid'].isin(processed_transactions)].copy()

            if pdf_new.empty:
                print(" [STATUS] No new un-screened logs in the cloud; continuing to monitor...")
                time.sleep(3)
                continue

            print(f"[ALERT] Captured {len(pdf_new)} new un-screened banking logs. Firing pipeline...")

            # 6.1 Feature engineering (must match offline training)
            pdf_new['errorBalanceOrig'] = pdf_new['newbalanceOrig'] + pdf_new['amount'] - pdf_new['oldbalanceOrg']
            pdf_new['errorBalanceDest'] = pdf_new['newbalanceDest'] - pdf_new['oldbalanceDest']
            pdf_new['type_CASH_OUT'] = pdf_new['type'].apply(lambda x: 1 if x == 'CASH_OUT' else 0)
            pdf_new['type_TRANSFER'] = pdf_new['type'].apply(lambda x: 1 if x == 'TRANSFER' else 0)

            # 6.3 Build X_live using the EXACT names the scaler was fit on (case-insensitive match).
            #     Do NOT force lowercase — the model was trained on mixed casing.
            expected_cols = list(scaler.feature_names_in_)        # fallback: best_model.feature_names_in_
            available = {c.lower(): c for c in pdf_new.columns}
            X_live = pd.DataFrame(
                {col: pdf_new[available[col.lower()]].values for col in expected_cols},
                index=pdf_new.index,
            )

            # 6.4 Scale + predict
            X_live_scaled = scaler.transform(X_live)
            preds = best_model.predict(X_live_scaled)
            probs = best_model.predict_proba(X_live_scaled)[:, 1]

            # 6.5 Payload matches dbo.fraud_predictions COLUMN-FOR-COLUMN
            pdf_output = pd.DataFrame({
                'step':           pdf_new['step'],
                'type':           pdf_new['type'],
                'amount':         pdf_new['amount'],
                'nameOrig':       pdf_new['nameOrig'],
                'oldbalanceOrg':  pdf_new['oldbalanceOrg'],
                'newbalanceOrig': pdf_new['newbalanceOrig'],
                'nameDest':       pdf_new['nameDest'],
                'oldbalanceDest': pdf_new['oldbalanceDest'],
                'newbalanceDest': pdf_new['newbalanceDest'],
                'prediction':     preds.astype(int),
                'timestamp':      datetime.now(),
            })

            # 6.6 Write back via JDBC
            spark_output_df = spark.createDataFrame(pdf_output)
            (spark_output_df.write
             .format("jdbc")
             .option("url", jdbc_url)
             .option("dbtable", target_out_table)
             .option("user", db_properties.get("user"))
             .option("password", db_properties.get("password"))
             .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
             .mode("append")
             .save())

            processed_transactions.update(pdf_new['uid'].tolist())
            print(f"[SECURE] Screened successfully. Intercepted {int(sum(preds))} malicious fraud attacks!\n")

        except Exception as batch_error:
            print("\n [WARNING] Exception caught during this polling batch; isolating and retrying...")
            traceback.print_exc()   # full traceback so silent failures stop hiding

        time.sleep(3)

except KeyboardInterrupt:
    print("\n [STOP] PySpark inference engine has been safely deactivated by an administrator.")

 [INIT] PySpark AI Fraud Inference Engine is now online...
 Polling Amazon RDS for new incoming financial streams...



[ALERT] Captured 500 new un-screened banking logs. Firing pipeline...


[SECURE] Screened successfully. Intercepted 1 malicious fraud attacks!



[ALERT] Captured 13 new un-screened banking logs. Firing pipeline...


[SECURE] Screened successfully. Intercepted 0 malicious fraud attacks!



...

 [STATUS] No new un-screened logs in the cloud; continuing to monitor...


 [STATUS] No new un-screened logs in the cloud; continuing to monitor...


ERROR:root:KeyboardInterrupt while sending command.                 (0 + 1) / 1]
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt



 [STOP] PySpark inference engine has been safely deactivated by an administrator.


26/06/20 04:12:03 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 2464349 ms exceeds timeout 120000 ms
26/06/20 04:12:03 WARN SparkContext: Killing executors is not supported by current scheduler.
26/06/20 04:12:08 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$